# 🧠 CAR-IMU — Week 1 Pipeline
**Data preprocessing → Bio-PM token extraction → Baseline HAR evaluation**

Steps:
1. ⚙️ Setup (Colab or local)
2. 📦 Preprocess WISDM → Bio-PM HDF5 format
3. 🔬 Extract Bio-PM token features
4. 📊 Week 1 analysis: UMAP, NN sanity check, LOSO baseline

> **Baseline to establish: ~0.689 Macro-F1 (real-only LOSO)**

## 🌐 Colab Setup (skip if running locally)

In [ ]:
import os
ON_COLAB = 'google.colab' in str(get_ipython())

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/aviral23032002/bio-pm-guided-synthetic-imu-generation.git'
    !git clone {REPO_URL} /content/project
    %cd /content/project

    !pip install torch h5py scikit-learn matplotlib umap-learn scipy --quiet
    print('✅ Colab setup complete!')
else:
    os.chdir(os.path.dirname(os.path.abspath('week1_pipeline.ipynb')))
    print(f'Local — working dir: {os.getcwd()}')

---
## Step 1 — Download WISDM Dataset
Skip if you already have it locally.

In [ ]:
import os

if os.path.exists('WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt'):
    print('✅ Dataset already exists — skipping download.')
else:
    print('Downloading WISDM dataset...')
    !wget -q https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_v1.1.tar.gz
    !tar -xzf WISDM_ar_v1.1.tar.gz
    print('✅ Downloaded and extracted.')

# Verify
!wc -l WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt

---
## Step 2 — Preprocess WISDM → Bio-PM Format

Converts raw WISDM CSV to `Data_MeLabel_*.h5` files expected by the Bio-PM encoder.

What it does:
- Resamples 20Hz → 30Hz
- Extracts zero-crossing movement elements
- Saves one HDF5 per subject in `preprocessed_biopm/`

**Expected output:** 36 HDF5 files, one per subject

In [ ]:
!python preprocess_wisdm_biopm.py \
    --raw_data  WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt \
    --out_dir   preprocessed_biopm

# Verify output
import os
files = [f for f in os.listdir('preprocessed_biopm') if f.endswith('.h5')]
print(f'\n✅ Created {len(files)} subject HDF5 files')

---
## Step 3 — Extract Bio-PM Features

Runs the frozen Bio-PM encoder over all preprocessed subjects.

Produces:
- `features/biopm_features_all.npz` — 1028-d feature vectors (used for Week 1 baseline)
- `results_week1/token_store.hdf5` — full token matrices (192×64 per window, used for Week 2)

In [ ]:
# Extract 1028-d features for all subjects
!python CS690TR/scripts/extract_features.py \
    --data_dir   preprocessed_biopm \
    --checkpoint CS690TR/checkpoints/checkpoint.pt \
    --out_file   features/biopm_features_all.npz

# Verify
import numpy as np
data = np.load('features/biopm_features_all.npz')
print('\nFeature file keys:', list(data.keys()))
print('Features shape:', data['features'].shape)   # (N, 1028)
print('Labels shape:  ', data['labels'].shape)

---
## Step 4 — Week 1 Analysis

Runs all five Week 1 analyses in one script:
1. **Token extraction** → `token_store.hdf5` (full 192×64 token matrices)
2. **NN sanity check** → same-class nearest-neighbour accuracy
3. **UMAP** → activity-coloured token embedding plot
4. **Subject embeddings** → subject-style separation analysis
5. **LOSO baseline** → real-only Macro-F1

**Expected result: ~0.689 Macro-F1**

In [ ]:
!python week1_analysis.py \
    --data_dir   preprocessed_biopm \
    --checkpoint CS690TR/checkpoints/checkpoint.pt \
    --features   features/biopm_features_all.npz \
    --out_dir    results_week1

---
## Step 5 — View Results

In [ ]:
# Inspect token store
import h5py, numpy as np

with h5py.File('results_week1/token_store.hdf5', 'r') as f:
    print('Keys:', list(f.keys()))
    tokens  = f['tokens'][:]
    labels  = f['labels'][:]
    subj    = f['subject_ids'][:]

ACTIVITY = {0:'Walking',1:'Jogging',2:'Upstairs',3:'Downstairs',4:'Sitting',5:'Standing'}

print(f'\ntokens shape:      {tokens.shape}')   # (10810, 192, 64)
print(f'Unique labels:     {np.unique(labels).astype(int).tolist()}')
print(f'\nClass distribution:')
for cls_id, name in ACTIVITY.items():
    n   = (labels == cls_id).sum()
    pct = 100 * n / len(labels)
    bar = '█' * int(pct / 2)
    flag = ' ⚠ MINORITY' if pct < 10 else ''
    print(f'  {name:<12} {n:>5} ({pct:4.1f}%) {bar}{flag}')

In [ ]:
# Show UMAP plots saved by week1_analysis.py
from IPython.display import Image, display
import os

for fname in ['umap_activity.png', 'umap_subjects.png']:
    path = os.path.join('results_week1', fname)
    if os.path.exists(path):
        print(f'--- {fname} ---')
        display(Image(path))
    else:
        print(f'Not found: {path}')

In [ ]:
# Show LOSO baseline results
import numpy as np

results_path = 'results_week1/loso_results.npy'
if os.path.exists(results_path):
    res = np.load(results_path, allow_pickle=True).item()
    print('LOSO Baseline (real only, 1028-d features)')
    print('=' * 40)
    for clf_name, f1_list in res.items():
        f1_arr = np.array(f1_list)
        print(f'{clf_name:<15} Macro-F1: {f1_arr.mean():.3f} ± {f1_arr.std():.3f}')
else:
    print('Results file not found — run Step 4 first.')

---
## 📝 Week 1 Notes

### Key findings
| Analysis | Result | Meaning |
|---|---|---|
| NN sanity check | ~90% same-class accuracy | Tokens cluster well by activity |
| Subject separation | ~0% | No need for subject conditioning |
| LOSO Macro-F1 | **0.689** | Baseline to beat with augmentation |

### Class imbalance
| Class | Windows | % |
|---|---|---|
| Walking | 4168 | 38.6% |
| Jogging | 3361 | 31.1% |
| Upstairs | 1224 | 11.3% |
| Downstairs | 994 | 9.2% |
| Sitting | 592 | 5.5% |
| Standing | 471 | 4.4% |

### Design decisions made
- **Activity-only conditioning** for CAR-IMU (subject conditioning not needed)
- **Token space** (192×64) is the generation target, not raw IMU signals
- Focus augmentation on Sitting, Standing, Downstairs, Upstairs